<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Lane Filter

The Histogram filter is used to estimate your Duckiebot’s lane pose ($d$, $\phi$), where $d$ is its lateral deviation relative to the lane's center and $\phi$ is its angular deviation relative to the lane's orientation.

<!--
 JT: this notation is confusing. 

 1. it should be a d(t) or d_k, phi(t) or \phi_k - i.e. highlight time dependency
 2. it should be explained that it is the position (not "deviation") of the mid-axle point of the robot ("A" in previous episodes), not, e.g., of the center of mass of the robot
 3. \phi (used \theta before but whatever) is an "orientation" rather than an "angular deviation"
 4. Observation: according to modeling introduced in previous mooc classes, the robot frame (x_r, y_r) is defined with 
    a. direction x_r: longitudital axis (looking forward) 
    b. axis y_r: 90deg counter-clockwise w.r.t. x_r
    c. d_{r/l} were used to indicate the displacements of the right and left wheels respectively - which are both in the x_r direction in the robot frame (see pics below - feel free to remove after review)

Here instead "d" is used for the y_r direction (actually, in the world frame Y), and later on in the segments -> pose algorithm picture, d is defined as an "x" ["d^i=0.5(x_1+x_2)^i"], while really being a "Y"

To summarize, I would stick with the previously introduced notation and clarify across the board that "d" is a y
-->

<img
    src="../assets/images/robot-and-world-frames.jpg"
    width="400"
    style="display: block; margin-left: auto; margin-right: auto;"
/>

<img
    src="../assets/images/robot-odometry-model-notation.png"
    width="400"
    style="display: block; margin-left: auto; margin-right: auto;"
/>

<img
    src="../assets/images/pose-definition.jpg"
    width="400"
    style="display: block; margin-left: auto; margin-right: auto;"
/>

<!--
 ![duckiebot-pose](../assets/images/pose-definition.jpg)
-->

This probabilistic approach provides a robust way to localize your Duckiebot using noisy sensor data while accounting for uncertainty.

<!--
 JT: Probabilistic? up to now all models were deterministic. How do we go from there to here?  
-->

This filter maintains a discrete grid over the deviation values, where each grid cell represents a possible state of your Duckiebot.

<!--
 It doesn't maintain, it defines. And it's good for all Duckiebots, not just mine.
-->

This filter is *recursive*, meaning that it iterates between *predicting* the next state of your Duckiebot, using its known kinematic model and the data from its wheel encoders, and *updating*, based on the ground projected lane segments.

<!--
JT: It is fundamentally wrong to say that the prediction step uses measurements from wheel encoders. 
-->

## Prediction step

During the prediction step, the filter uses your Duckiebot's estimated linear and angular velocities to update the histogram, predicting how its lane pose evolves over time.

<!--
JT: This statement is meaningless - 1. "the histogram" is still undefined; 2. the prediction step does not use the estimates; it updates previous estimates using the kinematics model and commanded input
-->

The state transition is modeled based on your Duckiebot's kinematic model:

$$ \left\{ \begin{aligned}
d_t &= d_{t - 1} + v \Delta t \cos(\phi_{t - 1})\\
\phi_t &= \phi_{t - 1} + \omega \Delta t
\end{aligned}\right.
$$

where $v$ and $\omega$ are the linear and angular velocities, respectively, and $\Delta t$ is the time step.

<!--
JT: 
- $v$ and $\omega$ are the commanded input linear and angular velocities (v being constant in our implementation). 
- notation should indicate ($v$, $\omega$) at which time step (t-1)
- and $\Delta t$ is which time step exactly? It's a multirate system: of the commands, of the measurements (and which when considering wheel encoders and camera messages)
-->

The predicted state probabilities are updated by shifting the histogram values accordingly.

<!--
JT: this is a very uninformative statement: 
1. the above "process model" is deterministic, there are no probabilities involved (yet?)
2. the "shifting accordingly" part is jumble jumble. According to the model?  
-->

## Update step

Detected line segments are projected onto the ground plane and matched to the expected lane geometry.

<!--
JT: how does this matching work? It is not trivial, and there are underlying assumption. Reasons for it being non trivial: (a) straights / curves / intersections are assumed to all be straights? (b) there is uncertainty embedded in the implementation of the appearance specs of the city too, so we are assuming cities are built perfectly. What happens if various lane building assumption are broken?   
-->

For each segment:

* The filter computes votes for possible $(d, \phi)$ values based on the segment’s position and orientation.
* These votes are added to the corresponding bins in the histogram.

<!--
JT: (a) it's not really the "filter computing the votes". 
- there is a geometric model based on the apperance specs that, for each segment, produces a state estimate
- all these estimates are treated like different "votes". 
-->

This process updates the belief distribution, increasing confidence in states consistent with the observations.

<!--
JT: 
- this process updates the measurement model, not the belief
- multiplying (after normalizing) the measurement model with the prior belief produces a posterior belief, which then in turn updates the state estimate, by means of taking the mode of it (MAP estimate)  
-->

The filter’s best estimate of the lane pose corresponds to the histogram cell with the highest value (maximum likelihood).

<!--
JT: very confusing. It appears as if the estimate is taken directly from the measurement likelihood.
-->

This estimate is used to guide navigation.

<!--
JT: uninformative. We typically use the word "navigation" with a different meaning (something involving planning paths through intersections). I'd keep it easy with "driving", but then sure, everything we are doing here helps with driving. 
-->


The objective in this exercise is to build the functions that you will need to create your histogram filter. 

<!--
JT: excercise -> learning activity
histogram filter -> Histogram filter
-->

The histogram filter represents the state as a fixed set of hypotheses that correspond to the centroids of evenly spaced cells. 

<!--
JT: fixed set of hypotheses -> spatially discretized and quantized (?) probability function
-->

Each one of these hypothesis has an associated weight, and the weights should sum to one. 

<!--
JT: should -> must
-->

As a result, the histogram corresponds to a valid belief distribution over the state space. In math we can write this as a weighted sum of Dirac delta functions:

<!--
JT: valid? This is a definition, not a result.
-->

$bel(x_t) =  \sum_{i=1}^N w^i_t \delta(x_t - x^i)$

The API for this (and really any) filter will comprise three functions: `prior()`, `predict()` and `update()`. 

<!--
JT: 
- it would be nice to flesh out all the theory before jumping into the implementation.
- function names are a little confusing. In theory:
    - the "prior" estimate is such because it is created before receving measurements, by using the process model to "predict" what the next state will be
    - the "posterior estimate" is produced through the "update" step
    Maybe a better selection of names would be: 
    - predict() -> creates the prior
    - measure() -> created the measurement likelihood
    - update() -> multiplies the above and produces the posterior
-->

The `prior()` function sets up the initial belief weights, $w_0$ over the histogram. 

<!--
JT: not really. prior() will likely - still need to check the code - update the prior belief. If it's just doing the $w_0$ is it called only once? Then how about initialize() instead of prior()?
-->

The `predict()` function propagates forward the belief weights based on the motion model and the control, $u_t$. 

<!--
JT: there is still a key element missing. How to go from whatever model presented above to what this theory calls process model. There are (a) noises/uncertainties missing; (b) discretization steps missing

We need to really tie this up to the models previously presented:
- forward kinematics is the actual real model we should be considering
- dead reckoning (sp?) is somewhat of the transition model above, properly derived, where we can use the encoder data as well
- "transition model" above.  
-->

This amounts to propagating the centroids of each of the cells forward and then adding all of the weight up that lands in each bin. 

$\overline{bel}(x_t) =  \sum_{i=1}^N \sum_{j=1}^N w_{t-1}^j p(x^i|x^j,u_t) \delta(x_t - x^i)$

<!--
JT: this equation might be correct but it doesn't really make a lot of sense. It's just theory, and one that does not take into account the actual modeling done so far.
-->

In our case we will be using the odometry as a proxy for the control input so that we may use a simple kinematic model of the robot. 

<!--
JT: I guess so - but I'm not sure this is the right thing to do. Conceptually, encoders = measurements => should come in later in the game. Note that the encoder measurements carry a lot of uncertainty and errors too (wheel slipping, angular discretization, etc.)
-->

Finally, the `update()` function takes a measurement and uses it to update the weights of the histogram bins based on the incoming measurement, $z_t$. 

<!--
JT: Ugh. To make it clear, should be: "Update() multiplies the measurement likelihood with the prior distribution, and takes the state corresponding to the maximum modulus (max element)
-->

This is achieved by multiplying the weight in each bin by the likelihood that the measurement was generated by the state corresponding to the centroid of that bin:

$bel(x_t) = \sum_{i=1}^N \frac{\overline{w}^i_t p(z_t|x^i)}{\displaystyle\sum_{j=1}^N \overline{w}^j_t p(z_t|x^j)}\delta(x_t - x^i)$

<!--
JT: very convoluted explanation.  
-->

In this notebook we will proceed by loading one image and using the line detection and ground projection algorithms (we can consider them as a black box here) to detect all of the white and yellow line segments. Each line segment will contribute a vote in the measurement likelihood. 

<!--
JT: I though this notebook was about the Histogram filter? the detected segment -> estimate/vote is a small part of it.   
-->

We will define the state here to be the comprised of the distance from the center of the lane $d$ and the angle relative to the lane $\phi$. 

<!--
JT: Let the state be (d,\phi) 
-->

![](../assets/images/state.png)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# start by importing some things we will need
import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from scipy.ndimage.filters import gaussian_filter
from scipy.stats import entropy, multivariate_normal
from math import floor, sqrt

@dataclass
class LaneFilterParams:
    # grid: JT: state space discretization. Could be updated to be non uniform - more fine where there is higher probability of finding the bot (around 0,0). My guess of units is introduced below.
    d: np.ndarray
    phi: np.ndarray
    delta_d: float #m
    delta_phi: float #rad
    d_min: float #m
    d_max: float #m
    phi_min: float #rad
    phi_max: float #rad
    # road: JT: what measurement units are we using? m? cm? inches? ft? (my guesses below)
    linewidth_white: float #m 
    linewidth_yellow: float #m
    lanewidth: float #m
    # robot: JT: which units for each? (these number should be picked up from the robot's odometry calibration file)
    wheel_radius: float #m
    wheel_baseline: float #m
    encoder_resolution: int #rad
    # filter: JT: tada! these magic parameters have not been discussed anywhere in the theory so far. What are they? Ho do we choose them? What is "mask"
    sigma_d_mask: float
    sigma_phi_mask: float
    mean_d_0: float
    mean_phi_0: float
    sigma_d_0: float
    sigma_phi_0: float


In [ ]:
# Now let's load the image that we will use. Feel free to change it, 
# but the calibrations in the setup/calibrations folder should correspond to the robot
# that took the image
from matplotlib.pyplot import imshow
%matplotlib inline
img = cv2.imread("../assets/images/pic1.png")
imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))

In [ ]:
import yaml
with open("../packages/dt-core/packages/lane_filter/config/lane_filter_node/default.yaml") as f:
    hp = yaml.safe_load(f)["lane_filter_histogram_configuration"]

with open("../packages/dt-core/packages/robots/duckiebot/dagu_car/config/kinematics_node/default.yaml") as f:
    kp = yaml.safe_load(f)

d, phi = np.mgrid[hp['d_min'] : hp['d_max'] : hp['delta_d'], hp['phi_min'] : hp['phi_max'] : hp['delta_phi']]

params = LaneFilterParams(
    d=d, phi=phi,
    delta_d=hp['delta_d'], delta_phi=hp['delta_phi'],
    d_min=hp['d_min'], d_max=hp['d_max'],
    phi_min=hp['phi_min'], phi_max=hp['phi_max'],
    linewidth_white=hp['linewidth_white'],
    linewidth_yellow=hp['linewidth_yellow'],
    lanewidth=hp['lanewidth'],
    wheel_radius=kp['radius'],
    wheel_baseline=kp['baseline'],
    encoder_resolution=135,
    sigma_d_mask=hp['sigma_d_mask'],
    sigma_phi_mask=hp['sigma_phi_mask'],
    mean_d_0=hp['mean_d_0'], mean_phi_0=hp['mean_phi_0'],
    sigma_d_0=hp['sigma_d_0'], sigma_phi_0=hp['sigma_phi_0'],
)
belief = np.empty(d.shape)


In [ ]:
def histogram_prior(belief, params):
    pos = np.empty(belief.shape + (2,))
    pos[:, :, 0] = params.d
    pos[:, :, 1] = params.phi
    RV = multivariate_normal(
        [params.mean_d_0, params.mean_phi_0],
        [[params.sigma_d_0, 0], [0, params.sigma_phi_0]]
    )
    return RV.pdf(pos)


In [ ]:
def histogram_predict(belief, left_encoder_ticks, right_encoder_ticks, params): #note that left/right_encoder_ticks are (should be) intended as Delta_ticks
    belief_in = belief

    alpha = 2 * np.pi / params.encoder_resolution
    d_left = params.wheel_radius * alpha * left_encoder_ticks 
    d_right = params.wheel_radius * alpha * right_encoder_ticks
    d_A = (d_left + d_right) / 2
    w = (d_right - d_left) / params.wheel_baseline # JT: 1. careful with how wheel_baseline is defined. In the MOOC slides wheel baseline = 2L, here we're assuming it is just =L; 2. Note that this is a Delta_phi, not a "w" (omega)

    maxids = np.unravel_index(belief.argmax(), belief.shape) #JT: what's up here? 
    phi_est = params.phi_min + (maxids[1] + 0.5) * params.delta_phi 
    v = d_A * np.sin(w + phi_est) 
    #JT: 1. bad choice of variable name: why call this "v" when it is a Delta_y_k; 
    # 2. "w" is a misnomer too, see comment above. 
    # 3. It makes more sense to do phi_est + delta_phi than the opposite.

    d_t = params.d + v
    phi_t = params.phi + w

    p_belief = np.zeros(belief.shape)

    for i in range(belief.shape[0]):
        for j in range(belief.shape[1]):
            if belief[i, j] > 0:
                if (
                    d_t[i, j] > params.d_max #JT: defining parameters out of the loop, and calling them directly might speed up this loop?
                    or d_t[i, j] < params.d_min
                    or phi_t[i, j] < params.phi_min
                    or phi_t[i, j] > params.phi_max
                ):
                    continue
                i_new = int(floor((d_t[i, j] - params.d_min) / params.delta_d))
                j_new = int(floor((phi_t[i, j] - params.phi_min) / params.delta_phi))
                p_belief[i_new, j_new] += belief[i, j]

    s_belief = np.zeros(belief.shape)
    gaussian_filter(p_belief, [params.sigma_d_mask, params.sigma_phi_mask], output=s_belief, mode='constant') # how are we choosing the gaussian sigmas, and how relevant is this choice?

    if np.sum(s_belief) == 0:
        return belief_in
    return s_belief / np.sum(s_belief)


<!--
JT: This section (how to go from segments -> pose estimate) is probably best as standalone
-->

Now we are going to work on building the measurement likelihood. We will have as an input a list of segments. Each segment has endpoints, a normal vector, and an associated color. 

<!--
JT: how is the normal defined? It's not shown in the pic below
-->

For each segment, we use geometric considerations to find what lateral position ($d$) and orientation ($\phi$) the robot would have had to have been at to detect the specific segment assuming that it did in fact come from a road marking. 

There is a bit of annoying detail here since the Duckiebot can detect lines on either side of the actual lane markings. We use the normals to determine which side of the lane marking the line was on. 

<!--
JT: how are the normals used?
-->

The following shows the `lanewidth` and the `linewidth_yellow` and `linewidth_white` parameters.

The following image shows a representations of how the detected line segments sit on an actual lane.

<p style="text-align: center">
  <img src="../assets/images/detected_line_segments.png" alt="detected line segments" width="200">
</p>

In [ ]:
# We will start by doing a little bit of processing on the segments to remove anything that is behind the robot (why would it be behind?)
# or a color not equal to yellow or white

def prepare_segments(segments):
    filtered_segments = []
    for segment in segments:

        # we don't care about RED ones for now
        if segment.color != SegmentColor.WHITE and segment.color != SegmentColor.YELLOW:
            continue
        # filter out any segments that are behind us
        if segment.points[0].x < 0 or segment.points[1].x < 0:
            continue

        filtered_segments.append(segment)
    return filtered_segments

Now for each segment we will generate a vote according to:

<!--
JT: I think the notation in the algorithm screenshot is confusing. In particular, in the robot frame - according to the theory presented in the MOOC - "x" is straight and "y" is lateral. Here instead "d" is a "x", while it should be a "y". 
-->

<p style="text-align: center">
  <img src="../assets/images/Votingalgorithm.png" alt="detected line segments" width="1000">
</p>

In [ ]:
def generate_vote(segment, params):
    p1 = segment.points[0].as_array()
    p2 = segment.points[1].as_array()
    t_hat = (p2 - p1) / np.linalg.norm(p2 - p1)

    n_hat = np.array([-t_hat[1], t_hat[0]])
    d1 = np.inner(n_hat, p1)
    d2 = np.inner(n_hat, p2)
    l1 = np.inner(t_hat, p1)
    l2 = np.inner(t_hat, p2)
    if l1 < 0:
        l1 = -l1
    if l2 < 0:
        l2 = -l2

    d_i = (d1 + d2) / 2
    phi_i = np.arcsin(t_hat[1])

    if segment.color == SegmentColor.WHITE:
        if p1[0] > p2[0]:  # right edge of white lane
            d_i -= params.linewidth_white
        else:              # left edge of white lane
            d_i = -d_i
            phi_i = -phi_i
        d_i -= params.lanewidth / 2
    elif segment.color == SegmentColor.YELLOW:
        if p2[0] > p1[0]:  # left edge of yellow lane
            d_i -= params.linewidth_yellow
            phi_i = -phi_i
        else:              # right edge of yellow lane
            d_i = -d_i
        d_i = params.lanewidth / 2 - d_i

    return d_i, phi_i


Now we generate the entire measurement likelihood by generating a vote for each line segment in the list that we received. The measurement likelihood will itself be a histogram:

<p style="text-align: center">
  <img src="../assets/images/Histogram.png" alt="detected line segments" width="600">
</p>

In [ ]:
def generate_measurement_likelihood(segments, params):
    measurement_likelihood = np.zeros(params.d.shape)

    for segment in segments:
        d_i, phi_i = generate_vote(segment, params)

        if d_i > params.d_max or d_i < params.d_min or phi_i < params.phi_min or phi_i > params.phi_max:
            continue

        i = int(floor((d_i - params.d_min) / params.delta_d))
        j = int(floor((phi_i - params.phi_min) / params.delta_phi))
        measurement_likelihood[i, j] += 1

    if np.linalg.norm(measurement_likelihood) == 0:
        return None
    measurement_likelihood /= np.sum(measurement_likelihood)
    return measurement_likelihood


Now we have everything we need for the update function. 

In [ ]:
def histogram_update(belief, segments, params):
    segmentsArray = prepare_segments(segments)
    measurement_likelihood = generate_measurement_likelihood(segmentsArray, params)

    if measurement_likelihood is not None:
        belief = np.multiply(belief, measurement_likelihood)
        if np.sum(belief) == 0:
            belief = measurement_likelihood
        else:
            belief = belief / np.sum(belief)
    return (measurement_likelihood, belief)


Now we have defined the `prior()`, `predict()` and `update()` functions. We will test one cycle of the filter here to see if things look reasonable. 

In [ ]:
belief = histogram_prior(belief, params)
imshow(belief)


In [ ]:
left = 10  # left ticks
right = 20  # right ticks
belief = histogram_predict(belief, left, right, params)
imshow(belief)


In [ ]:
from typing import Tuple, Dict, Union, List
from dt_computer_vision.camera import CameraModel, NormalizedImagePoint, Pixel
from dt_computer_vision.ground_projection import GroundProjector
from dt_computer_vision.ground_projection.rendering import draw_grid_image, debug_image
from dt_computer_vision.ground_projection.types import GroundPoint
from dt_computer_vision.line_detection import LineDetector, ColorRange, Detections
from dt_computer_vision.line_detection.rendering import draw_segments
from dt_state_estimation.lane_filter.types import Segment, SegmentColor, SegmentPoint




# This code will take the image that we loaded, detect the line segments, and project them onto the ground plane. 
# We don't need to worry too much about details here.
Color = Tuple[int, int, int]

camera_info: Dict[str, Union[np.ndarray, int]] = \
    {
        "width": 640,
        "height": 480,
        "K": np.reshape(
            [
                295.79606866959824,
                0.0,
                321.2621599038631,
                0.0,
                299.5389048862878,
                241.73616515312332,
                0.0,
                0.0,
                1.0,
            ],
            (3, 3)
        ),
        "D": [
            -0.23543978771661125,
            0.03637781479419574,
            -0.0033069818601306755,
            -0.0012140708179525926,
            0.0,
        ],
        "P": np.reshape(
            [
                201.14027404785156,
                0.0,
                319.5586620845679,
                0.0,
                0.0,
                239.74398803710938,
                237.60151004037834,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
            ],
            (3, 4)
        ),
        "H": np.reshape(
            [
                8.56148231e-03,
                2.22480148e-01,
                4.24318934e-01,
                -5.67022044e-01,
                -1.13258040e-03,
                6.81113839e-04,
                5.80917161e-02,
                4.35079347e+00,
                1.0],
            (3, 3)
        ),
    }
crop_top = 200
image_crop = [0, crop_top, 640, camera_info["height"] - crop_top]
x, y, w, h = image_crop
img_cropped = img[y:y + h, x:x + w, :]
_K = camera_info["K"]
_K[0][2] = _K[0][2] - x
_K[1][2] = _K[1][2] - y
# - update P
_P = camera_info["P"]
_P[0][2] = _P[0][2] - x
_P[1][2] = _P[1][2] - y


# colors
color_ranges: Dict[str, ColorRange] = {
    "white": ColorRange.fromDict({
        "low": [0, 0, 150],
        "high": [180, 100, 255]
    }),
    "yellow": ColorRange.fromDict({
        "low": [0, 100, 100],
        "high": [45, 255, 255]
    })
}
colors: Dict[str, Color] = {
    "red": (0, 0, 255),
    "yellow": (0, 255, 255),
    "white": (255, 255, 255),
}
color_order = ["yellow", "white"]
colors_to_detect = [color_ranges[c] for c in color_order]


In [ ]:

def detect_lines(img_cropped):
    
    detector = LineDetector()
    color_detections: List[Detections] = detector.detect(img_cropped, colors_to_detect)
    lines: Dict[str, dict] = {}
    for i, detections in enumerate(color_detections):
        color = color_order[i]
        # pack detections in a dictionary
        lines[color] = {
            "lines": detections.lines.tolist(),
            "centers": detections.centers.tolist(),
            "normals": detections.normals.tolist(),
            "color": color_ranges[color].representative
        }
    image_w_dets = draw_segments(img_cropped, {color_ranges["yellow"]: color_detections[0]})
    image_w_dets = draw_segments(image_w_dets, {color_ranges["white"]: color_detections[1]})
    plt.figure(0)
    imshow(cv2.cvtColor(image_w_dets,cv2.COLOR_BGR2RGB))
    return lines


In [ ]:
def lines_to_projected_segments(lines):
    camera = CameraModel(
        width=camera_info["width"],
        height=camera_info["height"],
        K=camera_info["K"],
        D=camera_info["D"],
        P=camera_info["P"],
        H=camera_info["H"],
    )
    projector = GroundProjector(camera)
    segments: List[Segment] = []
    colored_segments: Dict[Color, List[Tuple[GroundPoint, GroundPoint]]] = {}
    grid = draw_grid_image((400, 400))

    for color, colored_lines in lines.items():
        grounded_segments: List[Tuple[GroundPoint, GroundPoint]] = []
        for line in colored_lines["lines"]:
            # distorted pixels
            p0: Pixel = Pixel(line[0], line[1])
            p1: Pixel = Pixel(line[2], line[3])
            # distorted pixels to rectified pixels
            p0_rect: Pixel = camera.rectifier.rectify_pixel(p0)
            p1_rect: Pixel = camera.rectifier.rectify_pixel(p1)
            # rectified pixel to normalized coordinates
            p0_norm: NormalizedImagePoint = camera.pixel2vector(p0_rect)
            p1_norm: NormalizedImagePoint = camera.pixel2vector(p1_rect)
            # project image point onto the ground plane
            grounded_p0: SegmentPoint = projector.vector2ground(p0_norm)
            grounded_p1: SegmentPoint = projector.vector2ground(p1_norm)
            # add grounded segment to output
            segments.append(Segment(
                points=[grounded_p0, grounded_p1],
                color=SegmentColor(color)
            ))
            grounded_segments.append((grounded_p0, grounded_p1))

        colored_segments[colors[color]] = grounded_segments
    image_w_segs = debug_image(colored_segments, (400, 400), background_image=grid)
    image_w_segs_rgb = image_w_segs[:, :, [2, 1, 0]]
    plt.figure(1)
    imshow(image_w_segs_rgb)
    return segments



In [ ]:
%matplotlib inline

lines = detect_lines(img_cropped)
segments = lines_to_projected_segments(lines)
(measurement_likelihood, belief) = histogram_update(belief, segments, params)
plt.figure(2)
imshow(belief)


This process is repeated recursively as new encoder data and images arrive. The final state estimate that is reported is the $d$ and $\phi$ with the maximum likelihood (maximum value). 

In [ ]:
maxids = np.unravel_index(belief.argmax(), belief.shape)
d_max = params.d_min + (maxids[0] + 0.5) * params.delta_d
phi_max = params.phi_min + (maxids[1] + 0.5) * params.delta_phi
print(f"Current state estimate is d={d_max} and phi={phi_max}")

Now let's see how we can use this estimate to control the Duckiebot to follow the lane in the [lane control notebook](./04_lane_control.ipynb).